# Nacimientos DANE: Cuadros 3 de 2008–2025

Consulta del Parquet consolidado. Las columnas se ordenan así: `ano`, `codigo_departamento`, `nombre_departamento`, `codigo_municipio`, `nombre_municipio` y los conteos por sexo y área. El archivo conserva el nombre `births_2008.parquet`, pero contiene los dieciocho años. Las columnas de sexo indeterminado no presentes en una hoja original quedan nulas.

Para generarlo, ejecutar desde la raíz del repositorio:
```bash
.venv/bin/python scripts/processing/process_births_dane.py
```

Para 2008–2019 se excluyen las hojas `Indice` y `00`. Para 2020–2025 se lee `Cuadro3`, columnas C:T, y se excluye el total nacional. Los departamentos se propagan a sus municipios y se comprueba que el prefijo del código municipal coincida con el departamento. Los nombres de municipio no contienen el código. Los códigos se guardan como texto para conservar los ceros iniciales. Las filas `Total` departamentales tienen como código municipal el código del departamento. Todas las filas de `Extranjeros` usan `-1`; las de la hoja `Sin_información` y las filas municipales `SIN INFORMACIÓN` usan `-2`. Las categorías de extranjeros y sin información tienen código de departamento nulo.

Se conservan los totales y los desgloses del Excel. **Los valores son nacimientos, no población.** Las validaciones siguientes usan únicamente la columna `total`, sumando los desgloses por año y departamento/categoría y comparándolos con las filas `Total`. El script también realiza esta comprobación antes de escribir el Parquet.

Los archivos de 2024–2025 se leen como XLSX. Las etiquetas `Total` y `Total Dpto` se guardan como `Total` para conservar el esquema y comparar los desgloses de todos los años.

In [23]:
from pathlib import Path

import pandas as pd
import pyarrow.parquet as pq
from IPython.display import display

# Funciona desde la raíz del repositorio o desde notebooks/.
root = Path.cwd().resolve()
root = next(
    (path for path in (root, *root.parents)
     if (path / "scripts/processing/process_births_dane.py").is_file()),
    None,
)
if root is None:
    raise FileNotFoundError("Ejecute el notebook dentro del repositorio data-processing")

parquet_path = root / "data/processed/births_mortality_dane/births/births_2008.parquet"
if not parquet_path.is_file():
    raise FileNotFoundError(
        f"No existe {parquet_path}. Ejecute scripts/processing/process_births_dane.py"
    )

births = pd.read_parquet(parquet_path)
print(f"Archivo: {parquet_path}")
print(f"Filas: {len(births):,} | Columnas: {len(births.columns)}")

Archivo: /home/diego/repos/data-processing/data/processed/births_mortality_dane/births/births_2008.parquet
Filas: 21,802 | Columnas: 21


## Estructura y schema

In [24]:
print(pq.read_schema(parquet_path))

schema = pd.DataFrame({
    "columna": births.columns,
    "tipo_pandas": births.dtypes.astype(str).to_numpy(),
    "nulos": births.isna().sum().to_numpy(),
})
display(schema)

ano: int64
codigo_departamento: large_string
nombre_departamento: large_string
codigo_municipio: large_string
nombre_municipio: large_string
total: int64
total_hombres: int64
total_mujeres: int64
cabecera_municipal_hombres: int64
cabecera_municipal_mujeres: int64
centro_poblado_hombres: int64
centro_poblado_mujeres: int64
rural_disperso_hombres: int64
rural_disperso_mujeres: int64
sin_informacion_hombres: int64
sin_informacion_mujeres: int64
total_indeterminado: int64
cabecera_municipal_indeterminado: int64
centro_poblado_indeterminado: int64
rural_disperso_indeterminado: int64
sin_informacion_indeterminado: int64
-- schema metadata --
pandas: '{"index_columns": [], "column_indexes": [], "columns": [{"name":' + 2990


,columna,tipo_pandas,nulos
0,ano,Int64,0
1,codigo_departamento,string,444
2,nombre_departamento,string,0
3,codigo_municipio,string,0
4,nombre_municipio,string,0
5,total,Int64,0
6,total_hombres,Int64,0
7,total_mujeres,Int64,0
8,cabecera_municipal_hombres,Int64,0
9,cabecera_municipal_mujeres,Int64,0


## Primeras 10 filas con encabezados

In [25]:
with pd.option_context("display.max_columns", None, "display.max_colwidth", 80):
    display(births.head(10))

,ano,codigo_departamento,nombre_departamento,codigo_municipio,nombre_municipio,total,total_hombres,total_mujeres,cabecera_municipal_hombres,cabecera_municipal_mujeres,centro_poblado_hombres,centro_poblado_mujeres,rural_disperso_hombres,rural_disperso_mujeres,sin_informacion_hombres,sin_informacion_mujeres,total_indeterminado,cabecera_municipal_indeterminado,centro_poblado_indeterminado,rural_disperso_indeterminado,sin_informacion_indeterminado
0,2008,05,ANTIOQUIA,05,Total,90973,47139,43834,35271,32865,3555,3338,6800,6266,1513,1365,<NA>,<NA>,<NA>,<NA>,<NA>
1,2008,05,ANTIOQUIA,05001,MEDELLÍN,33228,17128,16100,16733,15711,208,184,95,95,92,110,<NA>,<NA>,<NA>,<NA>,<NA>
2,2008,05,ANTIOQUIA,05002,ABEJORRAL,309,167,142,53,39,6,7,100,90,8,6,<NA>,<NA>,<NA>,<NA>,<NA>
3,2008,05,ANTIOQUIA,05004,ABRIAQUÍ,27,15,12,4,2,0,1,11,9,0,0,<NA>,<NA>,<NA>,<NA>,<NA>
4,2008,05,ANTIOQUIA,05021,ALEJANDRÍA,69,38,31,18,15,0,0,18,15,2,1,<NA>,<NA>,<NA>,<NA>,<NA>
5,2008,05,ANTIOQUIA,05030,AMAGÁ,361,187,174,78,76,40,41,57,48,12,9,<NA>,<NA>,<NA>,<NA>,<NA>
6,2008,05,ANTIOQUIA,05031,AMALFI,411,210,201,101,88,9,11,96,96,4,6,<NA>,<NA>,<NA>,<NA>,<NA>
7,2008,05,ANTIOQUIA,05034,ANDES,650,345,305,135,111,75,68,119,108,16,18,<NA>,<NA>,<NA>,<NA>,<NA>
8,2008,05,ANTIOQUIA,05036,ÁNGELÓPOLIS,85,49,36,16,17,17,8,13,11,3,0,<NA>,<NA>,<NA>,<NA>,<NA>
9,2008,05,ANTIOQUIA,05038,ANGOSTURA,179,98,81,28,16,2,0,61,64,7,1,<NA>,<NA>,<NA>,<NA>,<NA>


## Validación de totales por año y departamento/categoría

Se suma exclusivamente la columna `total` de las filas distintas de `Total`, dentro de cada año y departamento/categoría, y se compara con el total publicado. Se incluyen las filas municipales sin información y los países extranjeros para reproducir el alcance del total.

En 2008–2019, Bogotá y la categoría sin información solo tienen una fila `Total` por año. En 2020–2025, Bogotá sí incluye desglose municipal; solo la categoría sin información carece de él. Los grupos sin detalle se muestran como **sin desglose**. Los códigos `-1` y `-2` pueden repetirse; la agrupación usa el año, el departamento y su nombre.

In [26]:
group_columns = ["ano", "codigo_departamento", "nombre_departamento"]
is_total = births["nombre_municipio"].str.casefold().eq("total")
reported = births.loc[is_total, group_columns + ["total"]].rename(
    columns={"total": "total_publicado"}
)
assert not reported.duplicated(group_columns).any(), "Hay más de un Total por grupo"

detail = births.loc[~is_total]
# dropna=False conserva las categorías sin código de departamento.
sums = detail.groupby(group_columns, dropna=False).agg(
    suma_desglose=("total", lambda values: values.sum(min_count=1)),
    filas_desglose=("total", "size"),
    valores_presentes=("total", "count"),
).reset_index()
validation = reported.merge(sums, on=group_columns, how="outer", indicator=True)
assert not validation["_merge"].eq("right_only").any(), "Hay desgloses sin fila Total"
validation["filas_desglose"] = validation["filas_desglose"].fillna(0).astype(int)
validation["diferencia"] = validation["suma_desglose"] - validation["total_publicado"]
validation["estado"] = "sin desglose"
has_detail = validation["filas_desglose"].gt(0)
comparable = (
    has_detail & validation["valores_presentes"].eq(validation["filas_desglose"])
    & validation["total_publicado"].notna()
)
validation.loc[has_detail | validation["total_publicado"].isna(), "estado"] = "datos incompletos"
validation.loc[comparable, "estado"] = "diferencia"
validation.loc[comparable & validation["diferencia"].eq(0), "estado"] = "coincide"
validation = validation.drop(columns=["_merge", "valores_presentes"])

with pd.option_context("display.max_rows", None, "display.max_columns", None):
    display(validation)

,ano,codigo_departamento,nombre_departamento,total_publicado,suma_desglose,filas_desglose,diferencia,estado
0,2008,05,ANTIOQUIA,90973,90973,126,0,coincide
1,2008,08,ATLÁNTICO,41432,41432,24,0,coincide
2,2008,11,BOGOTÁ,117590,<NA>,0,<NA>,sin desglose
3,2008,13,BOLÍVAR,30851,30851,47,0,coincide
4,2008,15,BOYACÁ,19193,19193,124,0,coincide
5,2008,17,CALDAS,12503,12503,28,0,coincide
6,2008,18,CAQUETÁ,7209,7209,17,0,coincide
7,2008,19,CAUCA,17319,17319,43,0,coincide
8,2008,20,CESAR,21052,21052,26,0,coincide
9,2008,23,CÓRDOBA,26228,26228,31,0,coincide


In [27]:
print("Comparación de total por año y departamento/categoría:")
display(validation.groupby(["ano", "estado"]).size().unstack(fill_value=0))

issues = validation.loc[validation["estado"].isin(["diferencia", "datos incompletos"])]
if not issues.empty:
    display(issues)

# Usa solo las filas Total para evitar duplicar los desgloses.
print("Nacimientos publicados por año (suma de las filas Total):")
display(reported.groupby("ano")["total_publicado"].sum().to_frame())
assert issues.empty, "Hay diferencias o datos incompletos: revise la tabla de validación"
print("Todas las comparaciones con desglose coinciden con los totales publicados.")

Comparación de total por año y departamento/categoría:


estado,coincide,sin desglose
ano,,
2008,33,2
2009,33,2
2010,33,2
2011,33,2
2012,33,2
2013,33,2
2014,33,2
2015,33,2
2016,33,2


Nacimientos publicados por año (suma de las filas Total):


,total_publicado
ano,
2008,715453
2009,699775
2010,654627
2011,665499
2012,676835
2013,658835
2014,669137
2015,660999
2016,647521


Todas las comparaciones con desglose coinciden con los totales publicados.
